In [ ]:
import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

model = "llama-3.3-70b-versatile"

In [ ]:
SYSTEM_PROMPT = """
You are a helpful personal assistant.
Use your tools when you need real data.

When using tools:
- Choose the appropriate tool for the information you need.
- Use the tool result to decide your next action.
- You may call additional tools if needed.
- Give the user a concise final answer.

Do not reveal private chain-of-thought reasoning.
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
def check_calendar(day):
    return "10am: Standup with Alice, 2pm: Dentist"

In [ ]:
def search_contacts(name):
    contacts = {
        "Alice": "alice@example.com",
        "Bob": "bob@example.com"
    }

    return contacts.get(
        name,
        f"No contact found for {name}"
    )

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_calendar",
            "description": "Check calendar events for a given day.",
            "parameters": {
                "type": "object",
                "properties": {
                    "day": {
                        "type": "string",
                        "description": "The day to check, for example Thursday"
                    }
                },
                "required": ["day"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_contacts",
            "description": "Search for the contact details of a person by name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The name of the person"
                    }
                },
                "required": ["name"]
            }
        }
    }
]

In [ ]:
def execute_tool(name, args):

    if name == "check_calendar":
        return check_calendar(**args)

    if name == "search_contacts":
        return search_contacts(**args)

    return f"Unknown tool: {name}"

In [ ]:
def run_agent(messages):

    while True:

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason
        msg = response.choices[0].message

        messages.append(msg)

        # Final answer
        if finish_reason == "stop":
            return msg.content

        # Agent wants to use tools
        if finish_reason == "tool_calls":

            for tc in msg.tool_calls:

                name = tc.function.name
                args = json.loads(tc.function.arguments)

                # Show the action
                print(f"Action: {name}({args})")

                # Execute the tool
                result = execute_tool(
                    name,
                    args
                )

                # Show the observation
                print(f"Observation: {result}")

                # Add observation to conversation
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })

In [ ]:
messages.append({
    "role": "user",
    "content": "What are my Thursday plans? Is 3pm free?"
})

answer = run_agent(messages)

print(f"\nFinal Answer: {answer}")

In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    }
]

In [ ]:
messages.append({
    "role": "user",
    "content": """
    Check my Thursday calendar.
    Find the contact details of the person in my standup.
    Tell me my schedule and their contact information.
    """
})

answer = run_agent(messages)

print(f"\nFinal Answer: {answer}")